# Init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType 
from pyspark.sql.functions import col, regexp_extract, when, regexp_replace, trim

In [0]:
RENAME_MAP = {
    'prd_id': 'product_id',
    'prd_key': 'product_key',
    'prd_nm': 'product_name',
    'prd_cost': 'product_cost',
    'prd_line': 'product_line',
    'prd_start_dt': 'product_start_date',
    'prd_end_dt': 'product_end_date'
}

# Read from bronze

In [0]:
df = spark.table("workspace.bronze.crm_prd_info")

# Data Transformation

Things to fix
1. Trim the strings
2. rename the columns
3. Add color column and move the color part from product name to it
4. Add size column and move size part from product name to it
5. Add capacity column for container-type products


## Trim the strings

In [0]:
for field in df.schema.fields: 
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

## Rename the columns

In [0]:
for old_col, new_col in RENAME_MAP.items():
    df = df.withColumnRenamed(old_col, new_col)

## Add product color column

In [0]:
color = regexp_extract(
    col("product_name"),
    r"(?i)\b(Black|Red|Blue|Silver|Yellow)\b",
    1
)

df = df.withColumn(
    "product_color",
    when(color != "", color).otherwise(None)
)

## Add product size column

In [0]:
size = regexp_extract(
    col("product_name"),
    r"-\s*(38|40|42|44|46|48|50|52|54|56|58|60|62|XL|S|M|L|Large)$",
    1
)

df = df.withColumn(
    "product_size",
    when(size == "Large", "L")
    .when(size != "", size)
    .otherwise(None)
)

## Add product capacity column for container products

In [0]:
capacity = regexp_extract(
    col("product_name"),
    r"(\d+)\s*oz\.?",
    1
)

df = df.withColumn(
    "product_capacity_oz",
    when(capacity != "", capacity.cast("int"))
    .otherwise(None)
)

## Remove size, capacity, and colors from name column

In [0]:
df = df.withColumn(
    "product_name",
    trim(
        regexp_replace(
            regexp_replace(
                regexp_replace(
                    col("product_name"),
                    r"(?i)\b(Black|Red|Blue|Silver|Yellow)\b",  # remove color
                    ""
                ),
                r"-\s*(38|40|42|44|46|48|50|52|54|56|58|60|62|XL|S|M|L|Large)$",  # remove size
                ""
            ),
            r"-?\s*\d+\s*oz\.?$",  # remove capacity
            ""
        )
    )
)

## Clean up the left over characters from product name

In [0]:
df = df.withColumn(
    "product_name",
    regexp_replace(col("product_name"), r"\s*-\s*$", "")
)

# Data Quality Checks

## Product Date Anomaly

200 source records have a `product_end_date` earlier than their
`product_start_date`.

Investigation of the Bronze/source data confirmed that these values already
exist in the original dataset and were not introduced by the Silver
transformation.

Since the intended business semantics of these dates are unknown, the original
values are preserved rather than applying an unsupported correction.

In [0]:
invalid_dates = df.filter(
    col("product_end_date").isNotNull()
    & (col("product_end_date") < col("product_start_date"))
)

print(f"Invalid date ranges: {invalid_dates.count()}")

In [0]:
df.display()

# Write it to silver tables

In [0]:
(
    df.write
    .mode("overwrite")
    .format("delta")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.silver.crm_prd_info")
)